# Install package

In [ ]:
# 1. Nang cap cong cu loi
!pip install -U pip "setuptools<82.0.0" "jedi>=0.16"

# 2. Cai dat he sinh thai Unsloth & Xformers
!pip install unsloth==2026.5.2 unsloth_zoo xformers


# Import Library

In [ ]:
import os
import re
import torch
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel

In [ ]:
DRIVE_DATA_DIR = ""
MODEL = "model_name_in_HF"
TRAIN_FILE = f"{DRIVE_DATA_DIR}/coder.jsonl"
VAL_FILE   = f"{DRIVE_DATA_DIR}/coder.eval.jsonl"


# Load Model

In [ ]:
# Bật tính năng Standby của Unsloth để tiết kiệm 30%+ bộ nhớ cho RL
os.environ['UNSLOTH_VLLM_STANDBY'] = "1"

max_seq_length = 2048 # Có thể tăng lên nếu suy luận dài
lora_rank = 32 # Rank cao hơn = thông minh hơn, nhưng train chậm hơn

# Tải mô hình cơ sở
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL, # Tên model LLM
    max_seq_length = max_seq_length,  # độ dài của context length mà mô hình có thể đọc và sinh ra trong 1 chu kỳ
    load_in_4bit = True,               # quantization xuống 4-bit
    fast_inference = False,           # tắt suy luận nhanh để ổn định hệ thống
    max_lora_rank = lora_rank,        # rank của LoRA
    load_in_fp8 = False,               # tắt chế độ tải mô hình bằng định dạng độ chính xác Float8
)
# Cấu hình LoRA Adapter
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # chọn module của model để áp dụng LoRA
    lora_alpha = lora_rank * 2,
    use_gradient_checkpointing = "unsloth",  # dọn dẹp bộ nhớ rác sinh ra khi train giảm VRAM
    random_state = 3407,
)

# Load Dataset

In [ ]:
from datasets import load_dataset

train_dataset = load_dataset("json", data_files=TRAIN_FILE, split="train")

try:
    val_dataset = load_dataset("json", data_files=VAL_FILE, split="train")
    print(f"Val dataset: {len(val_dataset)} samples")
except Exception as e:
    val_dataset = None
    print(f"No val dataset found ({e}), skipping.")

print(f"Train dataset: {len(train_dataset)} samples")
print(f"Sample messages: {train_dataset[0]['messages'][:2]}")

# Prepare Fine-Tune

In [ ]:
# Cấu hình thông tin chuẩn
YOUR_HF_TOKEN = "YOUR_HF_TOKEN" 
YOUR_HF_REPO = "YOUR_HF_REPO"
CHECKPOINT_REPO = "CHECKPOINT_REPO" # Repo phụ lưu nháp

In [ ]:
# Fix Unsloth hardcode eos/pad token bang cach patch file TRL
import trl, os
from unsloth.chat_templates import train_on_responses_only

sft_path = os.path.join(os.path.dirname(trl.__file__), "trainer", "sft_trainer.py")
with open(sft_path, "r") as f:
    code = f.read()

# Comment out 2 doan check eos_token va pad_token trong vocab
code = code.replace(
    'raise ValueError(\n                    f"The specified `eos_token`',
    'pass  # patched\n                if False: raise ValueError(\n                    f"The specified `eos_token`'
)
code = code.replace(
    'raise ValueError(\n                    f"The specified `pad_token`',
    'pass  # patched\n                if False: raise ValueError(\n                    f"The specified `pad_token`'
)

with open(sft_path, "w") as f:
    f.write(code)

print("Patched sft_trainer.py OK")

# Reload module
import importlib
importlib.reload(trl.trainer.sft_trainer)
from trl import SFTConfig, SFTTrainer



training_args = SFTConfig(
    output_dir = "/kaggle/working/",
    push_to_hub = True,                                
    hub_model_id = CHECKPOINT_REPO,             # Chỉ đẩy checkpoint nháp lên repo phụ này
    hub_token = YOUR_HF_TOKEN,                  
    hub_strategy = "checkpoint",                       
    save_steps = 50,                            # Tăng lên 50 hoặc 100 step để không bị nghẽn mạng do upload liên tục
    save_total_limit = 2,
    
    learning_rate = 2e-4,
    weight_decay = 0.01,
    bf16 = False,
    fp16 = True,
    max_grad_norm = 0.3,
    warmup_steps = 10,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",

    logging_steps = 1,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,

    num_train_epochs = 3,
    max_length = 2048,

    dataset_text_field = None,
    packing = False,

    report_to = "none",
    seed = 3407,
)

def formatting_func(example):
    msgs = example["messages"]
    if len(msgs) > 0 and isinstance(msgs[0], dict):
        text = tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=False
        )
        return [text]
    return [
        tokenizer.apply_chat_template(
            m, tokenize=False, add_generation_prompt=False
        )
        for m in msgs
    ]


trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    args = training_args,
    train_dataset = train_dataset,
    eval_dataset = val_dataset,
    formatting_func = formatting_func,
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)


# Fine-Tune

In [ ]:
import os
os.environ["ACCELERATE_MIXED_PRECISION"] = "no"

In [ ]:
trainer.train()
# trainer.train(
#     resume_from_checkpoint="checkpoint"
# )

# Upload the model to Hugging Face

In [ ]:
print("--- BƯỚC 1: LƯU TRỮ VÀ PUSH BẢN LORA ADAPTER (NHẸ) ---")
# Lưu cục bộ vào Kaggle working đề phòng sự cố mạng
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# Đẩy adapter lên Hugging Face Hub (Dung lượng nhỏ, chỉ vài chục MB)
model.push_to_hub(
    repo_id = YOUR_HF_REPO, 
    token = YOUR_HF_TOKEN, 
    private = False
)
tokenizer.push_to_hub(
    repo_id = YOUR_HF_REPO, 
    token = YOUR_HF_TOKEN
)
print("👉 Đã đẩy thành công bản Lora Adapter lên HF Hub!")


print("\n--- BƯỚC 2: MERGE THÀNH BẢN 16BIT VÀ PUSH LÊN HUB (TÙY CHỌN) ---")
# Nếu bạn muốn lưu cả bản Unsloth Merge 16bit nguyên bản không nén:
# model.push_to_hub_merged(YOUR_HF_REPO, tokenizer, save_method = "merged_16bit", token = YOUR_HF_TOKEN)


print("\n--- BƯỚC 3: TIẾN HÀNH MERGE, QUANTIZE Q4_K_M VÀ STREAM GGUF ---")
import sys
# Kích hoạt llama.cpp nội bộ tích hợp trong Unsloth
sys.path.insert(0, "/root/.unsloth/llama.cpp")

print("Hệ thống đang chạy nén trực tiếp trên RAM, không tốn dung lượng ổ cứng vật lý của Kaggle...")
print("Quá trình đóng gói và upload tự động sẽ mất khoảng 15-20 phút. Vui lòng giữ tab trình duyệt hoạt động.")

model.push_to_hub_gguf(
    repo_id = YOUR_HF_REPO,
    tokenizer = tokenizer,
    quantization_method = "q4_k_m",  # Chuẩn nén tối ưu hiệu năng/dung lượng tốt nhất hiện tại
    token = YOUR_HF_TOKEN,
    private = False 
)

print(f"\n🎉 HOÀN THÀNH XUẤT SẮC! File GGUF của bạn đã sẵn sàng tại: https://huggingface.co/{YOUR_HF_REPO}")